# Lesson 11.1 - AI Product Management Foundations (templates & metrics demo)

This notebook turns AI PM theory into reusable working assets:

- AI PRD template for product teams,
- offline model metric evaluation,
- translation from model metrics to product metrics,
- optional A/B-test style decision helper.

## Objectives

1. Build a practical AI PRD structure.
2. Compute precision/recall/F1/AUC on a toy product problem.
3. Connect technical metrics to product outcomes and launch decisions.

In [1]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

## AI PRD Template (Reusable)

Use this template as a baseline for AI feature PRDs.

### 1) Background and Problem
- Business context:
- User segments:
- Current pain and measurable impact:

### 2) AI Opportunity and Alternatives
- Why AI is considered:
- Non-AI alternatives explored:
- Decision rationale:

### 3) Data and Model Scope
- Data sources and ownership:
- Label quality and expected issues:
- Model family candidates:
- Known constraints (privacy/compliance/latency):

### 4) UX and Behavior Requirements
- User stories and acceptance criteria:
- Expected behavior under low confidence:
- Explainability requirements:
- Human-in-the-loop points:

### 5) Metrics and Launch Criteria
- Offline metrics (e.g., precision, recall, AUC):
- Online metrics (task success, CTR, retention, deflection):
- Guardrail metrics (complaints, unsafe output rate, override rate):
- Go/No-Go thresholds:

### 6) Risks and Mitigations
- Fairness risks:
- Abuse/misuse risks:
- Incident response plan:

## Create a Toy AI Product Dataset

Example scenario: a fraud-risk classifier that triages transactions for analyst review.

In [2]:
X, y = make_classification(
    n_samples=2500,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    weights=[0.8, 0.2],
    class_sep=1.0,
    random_state=SEED,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba),
}

pd.Series(metrics).sort_index()

accuracy     0.894400
f1           0.727273
precision    0.771930
recall       0.687500
roc_auc      0.914078
dtype: float64

## Inspect Confusion Matrix

In [3]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
cm_df

,Pred 0,Pred 1
Actual 0,471,26
Actual 1,40,88


## Mapping Offline Metrics to Product Metrics

Offline metrics do not automatically imply user or business success. Build an explicit mapping.

In [4]:
metric_mapping = pd.DataFrame(
    [
        {
            "offline_metric": "Recall",
            "product_proxy": "Fraud capture rate",
            "business_effect": "Higher fraud prevented",
            "risk_if_overoptimized": "Too many false alerts if precision drops",
        },
        {
            "offline_metric": "Precision",
            "product_proxy": "Analyst review efficiency",
            "business_effect": "Lower manual workload",
            "risk_if_overoptimized": "Missed fraud if recall drops",
        },
        {
            "offline_metric": "ROC-AUC",
            "product_proxy": "Ranking quality",
            "business_effect": "Better threshold flexibility",
            "risk_if_overoptimized": "Weak calibration may still hurt operations",
        },
    ]
)
metric_mapping

,offline_metric,product_proxy,business_effect,risk_if_overoptimized
0,Recall,Fraud capture rate,Higher fraud prevented,Too many false alerts if precision drops
1,Precision,Analyst review efficiency,Lower manual workload,Missed fraud if recall drops
2,ROC-AUC,Ranking quality,Better threshold flexibility,Weak calibration may still hurt operations


## Optional: Toy A/B Test Evaluation Stub

In [5]:
@dataclass
class ABResult:
    control_rate: float
    treatment_rate: float
    uplift: float


def evaluate_ab(control_successes: int, control_total: int, treatment_successes: int, treatment_total: int) -> ABResult:
    control_rate = control_successes / control_total
    treatment_rate = treatment_successes / treatment_total
    return ABResult(
        control_rate=control_rate,
        treatment_rate=treatment_rate,
        uplift=treatment_rate - control_rate,
    )


ab = evaluate_ab(control_successes=410, control_total=1000, treatment_successes=456, treatment_total=1000)
ab

ABResult(control_rate=0.41, treatment_rate=0.456, uplift=0.04600000000000004)

## Business / Research Case Studies & Exceptions

### Case 1: Fraud triage feature in fintech
- If recall improves but precision collapses, analysts burn out and operational cost spikes.
- Decision pattern: set a minimum precision guardrail while optimizing recall.

### Case 2: Recommendation quality in e-commerce
- Better offline ranking metrics may not improve revenue if UX placement is poor.
- Decision pattern: run interface-aware experiments, not model-only experiments.

### Case 3 (Exception): No-AI alternative wins
- A rules engine + better dashboard may deliver 80% of value at 20% complexity.
- Decision pattern: compare AI and non-AI baselines before commitment.

## Interview Questions & Answers

1. **Q: How do you decide whether to build an AI feature?**
   **A:** By validating user pain, data signal, expected ROI, and operational risk versus simpler alternatives.

2. **Q: What makes an AI PRD different from a normal PRD?**
   **A:** It includes data assumptions, model behavior requirements, uncertainty handling, and monitoring plans.

3. **Q: Why can high AUC fail in production?**
   **A:** Poor calibration, distribution shift, and UX mismatch can break real-world outcomes.

4. **Q: How do precision and recall relate to product decisions?**
   **A:** They represent trade-offs between false positives and false negatives tied to user and business costs.

5. **Q: What is a guardrail metric?**
   **A:** A metric that limits harmful side effects while optimizing primary KPIs.

6. **Q: How do you launch an AI feature safely?**
   **A:** Staged rollout, monitoring, fallback flows, and clear rollback criteria.

7. **Q: How do you connect offline and online metrics?**
   **A:** By building a metric tree from model outputs to user behavior and business impact.

8. **Q: What is a common AI PM anti-pattern?**
   **A:** Shipping AI due to hype without baseline comparisons or observability.

9. **Q: How should AI PMs handle non-deterministic outputs?**
   **A:** Use confidence-aware UX, user controls, and escalation paths.

10. **Q: When is AI the wrong tool?**
   **A:** When deterministic product changes solve the same problem more reliably.

11. **Q: How would you define success for a GenAI assistant?**
   **A:** Task completion, groundedness, cost per task, latency, and user trust.
